# Supervised Binary Classification on CSI Windows **with AveCSI**

Train a 1D CNN on MATLAB-exported windows (`X_window_*.npy`) and apply channel-wise AveCSI normalization.

## 1) Setup & configuration

In [ ]:

import os, json
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers, models, regularizers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

tf.keras.backend.clear_session()
for g in tf.config.experimental.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(g, True)
    except: pass

print("TensorFlow:", tf.__version__)

BASE        = "/home/tonyliao/WIFI_SENSING_LOCATION/PreCNN_Dataset/"
TRAIN_ROOT  = os.path.join(BASE, "training_set")
VAL_ROOT    = os.path.join(BASE, "val_set")
TEST_ROOT   = os.path.join(BASE, "test_set")

CLASS_NAMES = ["Empty", "Stationary"]
NUM_CLASSES = len(CLASS_NAMES)

FORCE_LAYOUT = "TC"   # MATLAB exports [time, channels]
CROP_LEN     = None   # If None pad to max T; else force to this T

AVECSI_PATH = os.path.join(BASE, "avecsi_empty.npz")
MATLAB_AVE_NAME = "avecsi_M.npy"  # optional fallback

BATCH_SIZE   = 128
EPOCHS       = 40
LR           = 5e-4
WEIGHT_DECAY = 1e-4

OUT_DIR = "/home/tonyliao/WIFI_SENSING_LOCATION/runs_windows_supervised_avecsi"
os.makedirs(OUT_DIR, exist_ok=True)


TensorFlow: 2.19.0


## 2) Utilities

In [29]:

def list_window_files(root_dir: str, class_names):
    files, labels = [], []
    for ci, cname in enumerate(class_names):
        print("path:", cname)
        patt = os.path.join(root_dir, cname, "**", "X_window_*")
        fns = [f for f in glob(patt, recursive=True) if os.path.isfile(f)]
        if len(fns) == 0:
            print(f"[WARN] no windows for class '{cname}' under {root_dir}")
        files.extend(sorted(fns))
        labels.extend([ci]*len(fns))
    return files, np.array(labels, dtype=np.int32)

def load_np_window(path, force_layout="TC"):
    X = np.load(path)
    X = X.astype(np.float32)
    if X.ndim != 2:
        raise ValueError(f"Expected 2D window, got {X.shape} for {path}")
    if force_layout == "TC":
        X = X.T   # -> [C, T]
    return X

def compute_T_stats(paths, force_layout="TC", crop_len=None):
    T_list = []
    C_first = None
    for p in paths:
        X = np.load(p)
        if X.ndim != 2:
            continue
        if force_layout == "TC":
            C, T = X.shape[1], X.shape[0]
        else:
            C, T = X.shape
        if C_first is None:
            C_first = C
        T_list.append(T if crop_len is None else crop_len)
    if len(T_list) == 0:
        return C_first, None, None
    T_max = max(T_list) if crop_len is None else crop_len
    T_min = min(T_list) if crop_len is None else crop_len
    return C_first, T_min, T_max

def find_first_file(root, name):
    for r, _, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    return None

def derive_mu_sigma_from_M(path_M, C_expected=None):
    M = np.load(path_M)  # [bins, C]
    if M.ndim != 2:
        raise ValueError(f"{path_M} is not 2D; got shape {M.shape}")
    bins, C_M = M.shape
    if C_expected is not None and C_M != C_expected:
        print(f"[WARN] avecsi_M C={C_M} != expected C={C_expected}; ignoring {path_M}")
        return None, None
    mu = M.mean(axis=0, keepdims=True).T.astype(np.float32)             # [C,1]
    sigma = (M.std(axis=0, keepdims=True).T + 1e-6).astype(np.float32)  # [C,1]
    return mu, sigma

def compute_avecsi_from_empty_windows(empty_paths, force_layout="TC"):
    if len(empty_paths) == 0:
        raise RuntimeError("No empty windows to compute AveCSI.")
    X0 = load_np_window(empty_paths[0], force_layout=force_layout)  # [C, T0]
    C = X0.shape[0]
    s1 = np.zeros((C,), dtype=np.float64)
    s2 = np.zeros((C,), dtype=np.float64)
    n  = np.zeros((C,), dtype=np.int64)
    s1 += X0.sum(axis=1); s2 += (X0**2).sum(axis=1); n += X0.shape[1]
    for p in empty_paths[1:]:
        x = load_np_window(p, force_layout=force_layout)
        if x.shape[0] != C:
            raise RuntimeError(f"Channel mismatch while computing AveCSI: {p} has C={x.shape[0]} != {C}")
        s1 += x.sum(axis=1); s2 += (x**2).sum(axis=1); n += x.shape[1]
    mu = (s1 / np.maximum(n,1))[:, None].astype(np.float32)
    var = (s2 / np.maximum(n,1)) - (mu[:,0].astype(np.float64))**2
    var = np.maximum(var, 1e-12)
    sigma = (np.sqrt(var)[:, None] + 1e-6).astype(np.float32)
    return mu, sigma

def load_or_compute_avecsi(base, train_root, force_layout="TC", class_names=("Empty","Stationary")):
    empty_paths, _ = list_window_files(train_root, [class_names[0]])
    if len(empty_paths) == 0:
        raise RuntimeError("No 'empty' windows under training_set to compute AveCSI.")
    X0 = load_np_window(empty_paths[0], force_layout=force_layout)
    C_expected = X0.shape[0]

    if os.path.isfile(AVECSI_PATH):
        z = np.load(AVECSI_PATH); mu, sigma = z["mu"], z["sigma"]
        if mu.shape[0] == C_expected and sigma.shape[0] == C_expected:
            print(f"[AveCSI] Loaded from {AVECSI_PATH} with C={C_expected}")
            return mu.astype(np.float32), sigma.astype(np.float32)
        else:
            print(f"[WARN] {AVECSI_PATH} C mismatch; will try other sources.")

    M_path = find_first_file(base, MATLAB_AVE_NAME)
    if M_path is not None:
        mu, sigma = derive_mu_sigma_from_M(M_path, C_expected=C_expected)
        if mu is not None:
            np.savez(AVECSI_PATH, mu=mu, sigma=sigma)
            print(f"[AveCSI] Derived from {M_path} and saved to {AVECSI_PATH}")
            return mu, sigma

    print("[AveCSI] Computing from training_set/empty windows ...")
    mu, sigma = compute_avecsi_from_empty_windows(empty_paths, force_layout=force_layout)
    np.savez(AVECSI_PATH, mu=mu, sigma=sigma)
    print(f"[AveCSI] Saved to {AVECSI_PATH}")
    return mu, sigma

def build_array(paths, labels, mu=None, sigma=None, force_layout="TC", crop_len=None, pad_to=None):
    Xs = []
    for p in paths:
        x = load_np_window(p, force_layout=force_layout)   # [C, T]
        if mu is not None and sigma is not None:
            x = (x - mu) / (sigma + 1e-6)
        if crop_len is not None:
            C, T = x.shape
            if T > crop_len:
                s = max(0, (T - crop_len)//2); x = x[:, s:s+crop_len]
            elif T < crop_len:
                pad = np.zeros((C, crop_len), dtype=x.dtype); pad[:, :T] = x; x = pad
        Xs.append(x)
    if crop_len is None:
        assert pad_to is not None, "pad_to must be provided when crop_len is None"
        C = Xs[0].shape[0]
        out = np.zeros((len(Xs), C, pad_to), dtype=np.float32)
        for i, xi in enumerate(Xs):
            t = min(pad_to, xi.shape[1]); out[i, :, :t] = xi[:, :t]
    else:
        out = np.stack(Xs, axis=0)
    y = labels.copy()
    return out, y


## 3) Load lists, AveCSI, and arrays

In [30]:

train_paths, y_train = list_window_files(TRAIN_ROOT, CLASS_NAMES)
val_paths,   y_val   = list_window_files(VAL_ROOT,   CLASS_NAMES)
test_paths,  y_test  = list_window_files(TEST_ROOT,  CLASS_NAMES)

if len(train_paths)==0 or len(val_paths)==0 or len(test_paths)==0:
    raise RuntimeError("One of the splits has 0 files. Check your folder structure and file names.")

C_train, T_train_min, T_train_max = compute_T_stats(train_paths, FORCE_LAYOUT, CROP_LEN)
C_val,   T_val_min,   T_val_max   = compute_T_stats(val_paths,   FORCE_LAYOUT, CROP_LEN)
C_test,  T_test_min,  T_test_max  = compute_T_stats(test_paths,  FORCE_LAYOUT, CROP_LEN)

if CROP_LEN is None:
    T_pad = max(t for t in [T_train_max, T_val_max, T_test_max] if t is not None)
else:
    T_pad = CROP_LEN

print(f"Detected C: train={C_train}, val={C_val}, test={C_test}")
print(f"Using common T = {T_pad}")

mu, sigma = load_or_compute_avecsi(BASE, TRAIN_ROOT, force_layout=FORCE_LAYOUT, class_names=CLASS_NAMES)
assert mu.shape[0] == C_train, f"AveCSI C={mu.shape[0]} does not match data C={C_train}"
print("AveCSI shapes:", mu.shape, sigma.shape)

X_train, y_train = build_array(train_paths, y_train, mu=mu, sigma=sigma,
                               force_layout=FORCE_LAYOUT, crop_len=CROP_LEN, pad_to=T_pad)
X_val,   y_val   = build_array(val_paths,   y_val,   mu=mu, sigma=sigma,
                               force_layout=FORCE_LAYOUT, crop_len=CROP_LEN, pad_to=T_pad)
X_test,  y_test  = build_array(test_paths,  y_test,  mu=mu, sigma=sigma,
                               force_layout=FORCE_LAYOUT, crop_len=CROP_LEN, pad_to=T_pad)

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape,   y_val.shape)
print("Test: ", X_test.shape,  y_test.shape)

C = X_train.shape[1]
assert X_val.shape[1] == C and X_test.shape[1] == C, "Channel count mismatch across splits."


path: Empty
path: Stationary
path: Empty
path: Stationary
path: Empty
path: Stationary
Detected C: train=106, val=106, test=106
Using common T = 4
path: Empty
[AveCSI] Computing from training_set/empty windows ...
[AveCSI] Saved to /home/tonyliao/WIFI_SENSING_LOCATION/PreCNN_Dataset/avecsi_empty.npz
AveCSI shapes: (106, 1) (106, 1)
Train: (22293, 106, 4) (22293,)
Val:   (7431, 106, 4) (7431,)
Test:  (7431, 106, 4) (7431,)


## 4) Model & train

In [31]:

def build_backbone_1d(in_channels: int, T: int, hidden: int = 128, wd: float = 1e-4, name="csi_backbone"):
    inp = layers.Input(shape=(in_channels, T), name="csi_in")
    x = layers.Permute((2,1), name="permute_T_C")(inp)
    x = layers.Conv1D(hidden, 7, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(wd), name="conv1")(x)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.MaxPooling1D(2, name="pool1")(x)
    x = layers.Conv1D(hidden, 5, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(wd), name="conv2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.MaxPooling1D(2, name="pool2")(x)
    x = layers.Dropout(0.2, name="drop2")(x)
    x = layers.Conv1D(hidden*2, 1, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(wd), name="conv3")(x)
    x = layers.BatchNormalization(name="bn3")(x)
    h = layers.GlobalAveragePooling1D(name="gap")(x)
    h = layers.Dense(hidden, activation="relu", name="embed")(h)
    return models.Model(inp, h, name=name)

backbone = build_backbone_1d(in_channels=C, T=X_train.shape[2], hidden=128, wd=WEIGHT_DECAY)
logits   = layers.Dense(NUM_CLASSES, activation=None, name="logits")(backbone.outputs[0])
model    = models.Model(backbone.inputs[0], logits, name="clf_windows_avecsi")

model.compile(optimizer=keras.optimizers.Adam(learning_rate=LR),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")])

model.summary()

hist = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

plt.figure(); plt.plot(hist.history['acc']); plt.plot(hist.history['val_acc'])
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(['train','val']); plt.title('Accuracy')
plt.savefig(os.path.join(OUT_DIR, 'acc_curve.png')); plt.close()

plt.figure(); plt.plot(hist.history['loss']); plt.plot(hist.history['val_loss'])
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(['train','val']); plt.title('Loss')
plt.savefig(os.path.join(OUT_DIR, 'loss_curve.png')); plt.close()

model.save(os.path.join(OUT_DIR, "clf_windows_avecsi.h5"))
backbone.save(os.path.join(OUT_DIR, "encoder_windows_avecsi.h5"))
print("Saved models to", OUT_DIR)


I0000 00:00:1761733627.359071   53524 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22181 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "clf_windows_avecsi"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ csi_in (InputLayer)             │ (None, 106, 4)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ permute_T_C (Permute)           │ (None, 4, 106)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv1D)                  │ (None, 4, 128)         │        95,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 4, 128)         │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling1D)            │ (None, 2, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 2, 128)         │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 2, 128)         │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling1D)            │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv1D)                  │ (None, 1, 256)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3 (BatchNormalization)        │ (None, 1, 256)         │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling1D)    │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embed (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits (Dense)                  │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 245,378 (958.51 KB)

 Trainable params: 244,354 (954.51 KB)

...

I0000 00:00:1761733630.255918   53701 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


164/175 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - acc: 0.9848 - loss: 0.0818

2025-10-29 10:27:11.232089: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_935', 4 bytes spill stores, 4 bytes spill loads



175/175 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - acc: 0.9856 - loss: 0.0793 - val_acc: 1.0000 - val_loss: 0.0363
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - acc: 1.0000 - loss: 0.0349 - val_acc: 1.0000 - val_loss: 0.0330
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - acc: 1.0000 - loss: 0.0324 - val_acc: 1.0000 - val_loss: 0.0304
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - acc: 1.0000 - loss: 0.0298 - val_acc: 1.0000 - val_loss: 0.0278
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - acc: 1.0000 - loss: 0.0271 - val_acc: 1.0000 - val_loss: 0.0251
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - acc: 1.0000 - loss: 0.0245 - val_acc: 1.0000 - val_loss: 0.0225
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 1.0000 - loss: 0.0219 - val_acc: 1.0000 - val_loss: 0.0201
Epoch 8/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - acc: 1.0000 - loss: 0.0196 - val_acc: 0.9997 - val_loss: 0.0195
Epoch 9/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.9998 

Saved models to runs_windows_supervised_avecsi


## 5) Evaluate

In [32]:

logits = model.predict(X_test, verbose=0)
pred = np.argmax(logits, axis=1)
test_acc = (pred == y_test).mean()
print("Test Accuracy:", float(test_acc))

cm = confusion_matrix(y_test, pred)
print("Confusion Matrix:\n", cm)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(); plt.title('Confusion Matrix (Test)')
plt.savefig(os.path.join(OUT_DIR, 'cm_test.png')); plt.close()

rep = classification_report(y_test, pred, target_names=CLASS_NAMES)
print("\nClassification Report:\n", rep)
with open(os.path.join(OUT_DIR, 'classification_report.txt'), 'w') as f:
    f.write(rep)

summary = {
    "base": BASE,
    "classes": CLASS_NAMES,
    "channels": int(C),
    "T_common": int(X_train.shape[2]),
    "epochs": int(EPOCHS),
    "batch_size": int(BATCH_SIZE),
    "lr": float(LR),
    "weight_decay": float(WEIGHT_DECAY),
    "train_samples": int(X_train.shape[0]),
    "val_samples": int(X_val.shape[0]),
    "test_samples": int(X_test.shape[0]),
    "test_accuracy": float(test_acc),
    "avecsi_path": AVECSI_PATH,
}
with open(os.path.join(OUT_DIR, 'run_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print("Saved run summary and figures to:", OUT_DIR)


2025-10-29 10:27:32.336521: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_128', 4 bytes spill stores, 4 bytes spill loads



Test Accuracy: 1.0
Confusion Matrix:
 [[3738    0]
 [   0 3693]]

Classification Report:
               precision    recall  f1-score   support

       Empty       1.00      1.00      1.00      3738
  Stationary       1.00      1.00      1.00      3693

    accuracy                           1.00      7431
   macro avg       1.00      1.00      1.00      7431
weighted avg       1.00      1.00      1.00      7431

Saved run summary and figures to: runs_windows_supervised_avecsi
